In [1]:
# IMPORTS

from __future__ import annotations
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

C:\Users\Fcomm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class LLM:
    def __init__(self):
        model_name = "google/flan-t5-small"
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def ask(self, prompt):
        inputs = self.tokenizer(prompt, return_tensors="pt")
        outputs = self.model.generate(**inputs, max_length=50)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

# create instance
llm = LLM()

Loading tokenizer...


Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4434.47it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [14]:
#Original
prompt = """
You are an agent in a grid.

State:
Ahead: Wall
Left: Floor
Right: Wall
Diagonal Left: Floor
Diagonal Right: Wall

Goal: Choose the best action to reach the goal.

Actions: left, right, forward

Rules:
- Do not move into walls
- Prefer open paths

Answer with ONLY one word: left, right, or forward.
"""
response = llm.ask(prompt)
print(response)

forward


In [ ]:
#Originality.ai
prompt = """
You are a navigation agent operating in a 2D grid-based world.
Your ultimate mission is to reach the "goal tile".
You MUST do so via the most optimal sequence of moves.

**Your perceptive field is limited to 6 tiles around you:**
- **Ahead:** the tile directly in front of you
- **Left:** the tile directly to your left
- **Right:** the tile directly to your right
- **Diagonal Left:** the tile diagonally forward-left
- **Diagonal Right:** the tile diagonally forward-right
- You can also **turn in-place**, **right** or **left** (90 degrees). This can help you e.g. go backwards, by turning in-place twice then moving forward.

**Movement rules:**
- You CANNOT enter wall tiles
- You CAN move onto floor tiles

---

**Reasoning Protocol — follow each step in order:**

0. Build, in your working memory, a map of the place as you explore it.
1. **Assess immediate moves:** Which of the valid actions are physically possible right now?
2. **Project one step further:** For each possible move, what does the diagonal and secondary tile information suggest about what lies beyond?
3. **Evaluate goal proximity:** Which passable move most plausibly advances toward an unknown goal, given no walls are blocking that corridor?
4. **Select optimal action:** Choose the single action with the best forward progress potential while avoiding immediate dead ends.

**Output format:**
- State your reasoning concisely through each step
- End with a single clearly labeled action: **Action: `[action]`**

Apply a greedy-but-aware strategy: prioritize immediate progress toward the goal while using diagonal tile data to avoid committing to paths that dead-end within the next move.

---

**Current Environment State:**

| Direction | Tile |
|---|---|
| Ahead | Wall |
| Left | Floor |
| Right | Wall |
| Diagonal Left | Wall |
| Diagonal Right | Floor |
"""
response = llm.ask(prompt)
print(response)

**Assess immediate moves:** Moves that are not forward are considered to be passable.** Moves that are not forward are considered to be passable.** Moves that are not forward are considered to be passable


In [15]:
#Quill
prompt = """
You are an agent navigating a grid-based environment.
At each step, you receive information about the surroundings:
whether there is a wall or open floor directly ahead, to the left, to the right, and diagonally to the right. 
Your goal is to choose the best single action—move left, right, or forward—to progress toward your objective. 
You must never move into a wall and should always prefer open paths when deciding. 
Given a specific state describing what is ahead, left, right, and diagonally right,
respond with only one word: "left," "right," or "forward," representing the safest and most efficient move according to the rules.
"""
response = llm.ask(prompt)
print(response)

Forward


In [16]:
#GPT
prompt = """
State:
Ahead=Wall, Left=Floor, Right=Wall, DiagonalLeft=Floor, DiagonalRight=Wall

Best action?
Answer: 
"""
response = llm.ask(prompt)
print(response)

Best action is to get the ball back.


In [17]:
#Gemini Pro
prompt = """
Context: You are an agent navigating a grid. You must not move into a Wall. You must move to a Floor. 
Current State:
- Ahead: Wall
- Left: Floor
- Right: Wall
- Diagonal Left: Floor
- Diagonal Right: Wall

Question: Based on the rules and current state, which action should you take? Choose from: left, right, forward. Answer with ONLY one word.

Answer:
"""
response = llm.ask(prompt)
print(response)

left


In [ ]:
#Gemini Pro [EleutherAI/gpt-neo-125M]
prompt = """
Task: Determine the safe move (left, right, or forward).

Input: Ahead: Wall, Left: Wall, Right: Floor
Output: right
###
Input: Ahead: Floor, Left: Wall, Right: Wall
Output: forward
###
Input: Ahead: Wall, Left: Floor, Right: Wall
Output: left
###
Input: Ahead: Wall, Left: Floor, Right: Wall, Diagonal Right: Floor
Output:
"""

In [18]:
#Prompt cowboy
prompt = """
You are a navigation agent in a grid-based environment. Respond with exactly one word representing your next move.

**Current State:**
- Ahead: Wall
- Left: Floor
- Right: Wall
- Diagonal Left: Floor
- Diagonal Right: Wall

**Available Actions:** `left`, `right`, `forward`

**Rules:**
- Never move into a wall
- Always choose an open path (Floor) when available
- If multiple open paths exist, prefer the one most likely to make forward progress toward the goal

Based on the current state, respond with only one word: `left`, `right`, or `forward`.
"""
response = llm.ask(prompt)
print(response)

Forward


In [19]:
#Claude
prompt = """
Grid navigation. Choose one action.

Surroundings:
- Ahead: Wall
- Left: Floor
- Right: Wall
- Diagonal Left: Floor
- Diagonal Right: Wall

Rules: Avoid walls. Use open paths.

Best action (left/right/forward):
"""
response = llm.ask(prompt)
print(response)

Paths are a way to navigate the floor.


In [20]:
#Copilot
prompt ="""
You are an agent navigating a grid.

State:
Ahead: Wall
Left: Floor
Right: Wall
Diagonal Left: Floor
Diagonal Right: Wall

Goal: Choose the best action to reach the goal.

Actions: left, right, forward

Rules:
- Do not move into walls
- Prefer open paths

Answer with ONLY one word: left, right, or forward.
"""
response = llm.ask(prompt)
print(response)

right


In [21]:
#Preplexity
prompt = """
Prompt:
You are an agent in a grid.

State:
Ahead: Wall
Left: Floor
Right: Wall
Diagonal Left: Floor
Diagonal Right: Wall

Goal: Choose the best action to reach the goal.

Actions: left, right, forward

Rules:

Do not move into walls.

Prefer open paths.

Answer with only one word: left, right, or forward.
"""
response = llm.ask(prompt)
print(response)

Forward


In [22]:
#Meta
prompt = """
Given an agent in a grid with the following state: Ahead: Wall, Left: Floor, Right: Wall, Diagonal Left: Floor, Diagonal Right: Wall. 
The goal is to choose the best action (left, right, or forward) to reach a goal, avoiding walls and preferring open paths. 
Respond with only one word for the action. 
Action:
"""
response = llm.ask(prompt)
print(response)

Diagonal


In [ ]:
class LLMRunner:
    def __init__(self, model_name):
        self.model_name = model_name

        print(f"[INFO] Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            low_cpu_mem_usage=True
        )

        self.model.eval()

    def run(self, prompt, max_new_tokens=2, temperature=0.0, top_p=0.9):
        inputs = self.tokenizer(prompt, return_tensors="pt")
        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p
            )

        output_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        result = {
            "model": self.model_name,
            "timestamp": datetime.now().isoformat(),
            "input_prompt": prompt,
            "input_tokens": int(input_len),
            "output_text": output_text,
            "output_tokens": int(outputs.shape[1])
        }

        return result

    def save(self, result, filename="llm_results.jsonl"):
        with open(filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(result) + "\n")

In [7]:
runner = LLMRunner("EleutherAI/gpt-neo-1.3B")

result = runner.run(prompt)

print("LLM Answer")
print(result["output_text"])

[INFO] Loading model: EleutherAI/gpt-neo-1.3B


: 

In [26]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "EleutherAI/gpt-neo-125M"    # swap models here

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True
)

# your existing prompt
text = prompt  

# tokenize (NO truncation since you need full input)
inputs = tokenizer(text, return_tensors="pt")

print("Token length:", inputs["input_ids"].shape[1])

print("Generating...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2,  
        do_sample=True,
        temperature=0.1,
        top_p=0.9
    )

print("\n--- OUTPUT ---\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 8137.56it/s]
GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Token length: 107
Generating...

--- OUTPUT ---


Determine the safe move (left, right, or forward) to step on a Floor and avoid a Wall.

State: Ahead: Wall, Left: Wall, Right: Floor
Move: right

State: Ahead: Floor, Left: Wall, Right: Wall
Move: forward

State: Ahead: Wall, Left: Floor, Right: Wall
Move: left

State: Ahead: Wall, Left: Floor, Right: Wall, Diagonal Right: Floor
Move:

State
